In [1]:
# =========================================
# [셀 1] 프로젝트 공통 설정 (경로/패키지/기본 파라미터)
# 목적:
# - 노트북 어디서 실행해도 경로가 안 꼬이게 "기준 루트"를 고정
# - 데이터/결과 폴더 자동 생성
# - 이후 셀에서 재사용할 기본 파라미터(이벤트 윈도우 등) 정의
# =========================================

import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

# (선택) 다운로드/가격데이터: yfinance를 쓸 예정
# - 아직 실제 호출은 다음 셀부터 하겠소(1셀은 세팅만).
try:
    import yfinance as yf
except Exception as e:
    yf = None
    print("[경고] yfinance import 실패. 다음 셀에서 설치/대체 수단을 쓰겠소.")
    print("       에러:", repr(e))

warnings.filterwarnings("ignore")


# -----------------------------
# 1) 프로젝트 루트/폴더 구조
# -----------------------------
# 사용자 환경 기준:
# - 프로젝트 폴더: C:\QP2
# - 노트북 위치 : C:\QP2\notebooks
BASE_DIR = Path(r"C:\QP2")  # 폴더 주소 고정
NOTEBOOK_DIR = BASE_DIR / "notebooks"

# 데이터/산출물 폴더 (필요시 추가 확장)
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
OUT_DIR = BASE_DIR / "outputs"

for d in [DATA_DIR, RAW_DIR, PROC_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ BASE_DIR:", BASE_DIR)
print("✅ DATA_DIR:", DATA_DIR)
print("✅ OUT_DIR :", OUT_DIR)


# -----------------------------
# 2) 연구 주제: '실적 발표 전 모멘텀 유지' 기본 정의
# -----------------------------
# 여기서 말하는 규칙(가설)은 "발표 전까지"를 핵심으로 보겠소.
# - 이벤트(실적발표) 날짜를 E라 할 때,
#   (A) 사전 모멘텀 측정 구간: [E - pre_lookback, E - 1]
#   (B) 직전 유지/강화 관측 구간: [E - hold_window, E - 1]
#
# 예:
# - pre_lookback=20이면, 발표 전 20거래일 추세를 '사전 모멘텀'으로 정의
# - hold_window=5이면, 발표 직전 5거래일 동안 그 추세가 유지/강화되는지 본다

PRE_LOOKBACK = 20      # 사전 모멘텀 측정(거래일)
HOLD_WINDOW = 5        # 발표 직전 관측(거래일)
MIN_GAP_DAYS = 20      # 두 실적발표 간 최소 간격(너무 가까우면 이벤트 중첩으로 제외)

# 수익률 계산 방식(로그수익률/단순수익률)
USE_LOG_RET = True

# 거래비용/슬리피지(추후 백테스트로 확장할 때 사용)
# - 지금 단계(현상 검정)에서는 바로 쓰지 않아도 되지만,
#   설정만 미리 박아두면 연구가 백테스트로 자연히 이어지오.
TCOST_BPS = 5  # 5bp 가정 (필요시 조정)


# -----------------------------
# 3) 데이터 범위/종목 설정(초안)
# -----------------------------
# 실적발표일(earnings calendar)은 소스에 따라 제약이 크오.
# 1차는 yfinance의 earnings_dates를 시도하되, 부족하면 다른 소스로 갈아타면 되오.
#
# ※ 종목 리스트는 임시요. 다음 셀에서 S&P500/나스닥100 등으로 확장 가능.
TICKERS = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "NVDA",
]

# 가격 데이터 기간(넉넉히 확보; 실적 이벤트 윈도우 계산을 위해 앞뒤 여유 필요)
START_DATE = "2015-01-01"
END_DATE   = None  # None이면 오늘까지(yfinance 기준)

print("\n[기본 파라미터]")
print("TICKERS      :", len(TICKERS), "개")
print("PRE_LOOKBACK :", PRE_LOOKBACK)
print("HOLD_WINDOW  :", HOLD_WINDOW)
print("TCOST_BPS    :", TCOST_BPS)
print("START_DATE   :", START_DATE, "/ END_DATE:", END_DATE)


# -----------------------------
# 4) 헬퍼: 수익률 계산 함수(이후 셀에서 재사용)
# -----------------------------
def compute_returns(px: pd.Series, use_log: bool = True) -> pd.Series:
    """
    종가 시계열에서 일간 수익률을 계산하오.
    - use_log=True  : log return
    - use_log=False : simple return
    """
    px = px.astype(float)
    if use_log:
        return np.log(px).diff()
    else:
        return px.pct_change()


# -----------------------------
# 5) 저장 파일명 규칙(이후 셀에서 사용)
# -----------------------------
def path_prices(ticker: str) -> Path:
    return RAW_DIR / f"prices_{ticker}.parquet"

def path_earnings(ticker: str) -> Path:
    return RAW_DIR / f"earnings_{ticker}.parquet"

print("\n✅ 셀 1 완료: 환경/경로/기본 설정이 정리되었소.")


✅ BASE_DIR: C:\QP2
✅ DATA_DIR: C:\QP2\data
✅ OUT_DIR : C:\QP2\outputs

[기본 파라미터]
TICKERS      : 5 개
PRE_LOOKBACK : 20
HOLD_WINDOW  : 5
TCOST_BPS    : 5
START_DATE   : 2015-01-01 / END_DATE: None

✅ 셀 1 완료: 환경/경로/기본 설정이 정리되었소.


In [2]:
# =========================================
# [셀 2] 가격 데이터 + 실적발표일(Earnings Dates) 수집/캐시
# 목적:
# - yfinance로 종가(Adj Close) 패널 구축용 원천 데이터 확보
# - 실적발표일(earnings dates) 수집해 이벤트 연구의 뼈대 마련
# - parquet로 저장하여 재실행 시 즉시 로드(시간 절약)
# =========================================

import time

FORCE_REDOWNLOAD = False   # True면 기존 캐시 무시하고 다시 받음
PRICE_FIELD = "Adj Close"  # 기본은 Adj Close 권장(분할/배당 조정)


def download_prices_one(ticker: str,
                        start: str = START_DATE,
                        end: str | None = END_DATE,
                        force: bool = False) -> pd.DataFrame:
    """
    개별 종목 가격 다운로드 후 parquet 캐시하오.
    반환: DataFrame(index=Date, columns=[PRICE_FIELD])
    """
    fp = path_prices(ticker)
    if fp.exists() and (not force):
        df = pd.read_parquet(fp)
        # 최소 검증
        if PRICE_FIELD not in df.columns:
            raise ValueError(f"[{ticker}] 캐시 파일에 '{PRICE_FIELD}' 컬럼이 없소: {fp}")
        return df.sort_index()

    if yf is None:
        raise RuntimeError("yfinance가 import되지 않았소. 설치/환경을 먼저 확인하시오.")

    df_raw = yf.download(
        tickers=ticker,
        start=start,
        end=end,
        auto_adjust=False,     # Adj Close를 따로 쓰고 싶으니 False
        progress=False,
        threads=False
    )

    if df_raw is None or len(df_raw) == 0:
        raise RuntimeError(f"[{ticker}] 가격 다운로드 실패(빈 DF). 티커/기간을 확인하시오.")

    # yfinance 반환 형태가 단일/멀티인덱스가 섞여 나오는 일이 있어 정리하오.
    if isinstance(df_raw.columns, pd.MultiIndex):
        # ('Adj Close', 'AAPL') 같은 구조일 가능성
        if (PRICE_FIELD, ticker) in df_raw.columns:
            s = df_raw[(PRICE_FIELD, ticker)].copy()
        elif PRICE_FIELD in df_raw.columns.get_level_values(0):
            # 티커 레벨이 2번째가 아닐 수 있어 우회
            s = df_raw.xs(PRICE_FIELD, axis=1, level=0).iloc[:, 0].copy()
        else:
            raise RuntimeError(f"[{ticker}] '{PRICE_FIELD}'를 찾지 못했소. columns={df_raw.columns}")
        df = s.to_frame(PRICE_FIELD)
    else:
        if PRICE_FIELD not in df_raw.columns:
            raise RuntimeError(f"[{ticker}] '{PRICE_FIELD}'를 찾지 못했소. columns={df_raw.columns}")
        df = df_raw[[PRICE_FIELD]].copy()

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df.to_parquet(fp, index=True)
    return df


def download_earnings_dates_one(ticker: str,
                               limit: int = 200,
                               force: bool = False) -> pd.DataFrame:
    """
    yfinance에서 실적발표일을 가능한 방식으로 긁어오고 parquet 캐시하오.
    반환: DataFrame(columns=['earnings_date', 'ticker', ...가능한 추가 컬럼])
    """
    fp = path_earnings(ticker)
    if fp.exists() and (not force):
        df = pd.read_parquet(fp)
        return df

    if yf is None:
        raise RuntimeError("yfinance가 import되지 않았소. 설치/환경을 먼저 확인하시오.")

    t = yf.Ticker(ticker)

    # 1) 권장: get_earnings_dates (버전/환경 따라 동작 차이 가능)
    df_e = None
    try:
        df_tmp = t.get_earnings_dates(limit=limit)
        if df_tmp is not None and len(df_tmp) > 0:
            df_e = df_tmp.copy()
    except Exception:
        df_e = None

    # 2) 보조: calendar(최근/다음)에서 뽑기(대개 빈약)
    if df_e is None or len(df_e) == 0:
        try:
            cal = t.calendar
            # calendar는 형태가 들쭉날쭉하니 매우 보수적으로 처리하오
            # 가능한 경우: index에 'Earnings Date'가 있고 값이 Timestamp/리스트
            if cal is not None and len(cal) > 0:
                if "Earnings Date" in cal.index:
                    v = cal.loc["Earnings Date"].values
                    dates = []
                    for x in v:
                        if pd.isna(x):
                            continue
                        # 리스트/배열/단일 timestamp 모두 대비
                        if isinstance(x, (list, tuple, np.ndarray)):
                            dates.extend(list(x))
                        else:
                            dates.append(x)
                    dates = [pd.to_datetime(d) for d in dates if str(d) != "NaT"]
                    dates = sorted(set([d.normalize() for d in dates]))
                    if len(dates) > 0:
                        df_e = pd.DataFrame({"Earnings Date": dates})
        except Exception:
            df_e = None

    if df_e is None or len(df_e) == 0:
        # 빈 결과도 저장은 해 두겠소(매번 시도하며 시간을 태우지 않게)
        df_out = pd.DataFrame(columns=["earnings_date", "ticker"])
        df_out.to_parquet(fp, index=False)
        return df_out

    # 표준화: earnings_date 컬럼으로 통일
    df_e = df_e.reset_index()
    # yfinance는 index가 Earnings Date인 경우가 흔하오
    cand_cols = [c for c in df_e.columns if str(c).lower() in ["earnings date", "earnings_date", "date"]]
    if len(cand_cols) == 0 and "index" in df_e.columns:
        # reset_index로 'index'가 생긴 경우
        df_e["earnings_date"] = pd.to_datetime(df_e["index"])
    else:
        df_e["earnings_date"] = pd.to_datetime(df_e[cand_cols[0]])

    df_e["ticker"] = ticker
    df_e = df_e.dropna(subset=["earnings_date"]).copy()
    df_e["earnings_date"] = df_e["earnings_date"].dt.tz_localize(None).dt.normalize()
    df_e = df_e.sort_values("earnings_date").drop_duplicates(subset=["ticker", "earnings_date"])

    df_e.to_parquet(fp, index=False)
    return df_e


# ---------------------------------
# 실행: 전체 티커에 대해 가격 + 실적일 수집
# ---------------------------------
prices_map = {}
earnings_map = {}

t0 = time.time()
for i, tk in enumerate(TICKERS, 1):
    print(f"\n[{i}/{len(TICKERS)}] {tk} 수집 중...")

    # (A) 가격
    df_px = download_prices_one(tk, start=START_DATE, end=END_DATE, force=FORCE_REDOWNLOAD)
    prices_map[tk] = df_px
    print(f"  - 가격: {len(df_px):,} rows  ({df_px.index.min().date()} ~ {df_px.index.max().date()})")

    # (B) 실적발표일
    df_e = download_earnings_dates_one(tk, limit=200, force=FORCE_REDOWNLOAD)
    earnings_map[tk] = df_e
    print(f"  - 실적발표일: {len(df_e):,} events")

dt = time.time() - t0
print(f"\n✅ 셀 2 완료. 총 소요: {dt:.1f}초")
print("   prices_map / earnings_map에 결과가 들어있소.")



[1/5] AAPL 수집 중...
  - 가격: 2,783 rows  (2015-01-02 ~ 2026-01-27)
  - 실적발표일: 0 events

[2/5] MSFT 수집 중...
  - 가격: 2,783 rows  (2015-01-02 ~ 2026-01-27)
  - 실적발표일: 0 events

[3/5] AMZN 수집 중...
  - 가격: 2,783 rows  (2015-01-02 ~ 2026-01-27)
  - 실적발표일: 0 events

[4/5] GOOGL 수집 중...
  - 가격: 2,783 rows  (2015-01-02 ~ 2026-01-27)
  - 실적발표일: 0 events

[5/5] NVDA 수집 중...
  - 가격: 2,783 rows  (2015-01-02 ~ 2026-01-27)
  - 실적발표일: 0 events

✅ 셀 2 완료. 총 소요: 0.1초
   prices_map / earnings_map에 결과가 들어있소.


In [3]:
# =========================================
# [셀 3] 실적발표일 대체: SEC 10-Q/10-K 제출일(filing date)로 이벤트 구성
# 목적:
# - yfinance earnings dates가 0건인 환경에서도 이벤트 스터디를 가능케 함
# - 10-Q/10-K 제출일을 E로 정의하고, 이전 셀의 로직(events_df)을 그대로 재사용
# 산출:
# - RAW_DIR/sec_filings_{ticker}.parquet
# - earnings_map을 "filing events"로 대체(earnings_date 컬럼 유지)
# =========================================

import json
import time
import requests

SEC_TICKER_CIK_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_SUBMISSIONS_URL = "https://data.sec.gov/submissions/CIK{cik10}.json"

# SEC는 User-Agent가 사실상 필수요. 아무 값이나 쓰지 말고, 네 이메일/프로젝트명을 넣는 게 정석이오.
# 예: "QP Research (your_email@example.com)"
SEC_USER_AGENT = "QP Research (mth081844@gmail.com)"  # <- 여기만 네 것으로 바꾸시오.

def sec_get(url: str, session: requests.Session) -> dict:
    r = session.get(url, headers={"User-Agent": SEC_USER_AGENT, "Accept-Encoding": "gzip, deflate"})
    r.raise_for_status()
    return r.json()

def load_ticker_cik_map(session: requests.Session) -> dict:
    data = sec_get(SEC_TICKER_CIK_URL, session)
    # company_tickers.json은 { "0": {"ticker":"AAPL","cik_str":320193,...}, ... } 형태
    mp = {}
    for _, v in data.items():
        t = str(v.get("ticker", "")).upper()
        cik = int(v.get("cik_str"))
        if t:
            mp[t] = cik
    return mp

def fetch_filings_10q_10k(ticker: str, session: requests.Session, max_n: int = 200) -> pd.DataFrame:
    """
    10-Q, 10-K 제출일을 가져와 이벤트 DF로 만들겠소.
    반환 컬럼: ['earnings_date','ticker','form','filingDate','reportDate','accessionNumber']
    """
    t = ticker.upper()
    cik = TICKER_CIK.get(t)
    if cik is None:
        return pd.DataFrame(columns=["earnings_date","ticker","form","filingDate","reportDate","accessionNumber"])

    cik10 = str(cik).zfill(10)
    js = sec_get(SEC_SUBMISSIONS_URL.format(cik10=cik10), session)

    recent = js.get("filings", {}).get("recent", {})
    forms = recent.get("form", [])
    filing_dates = recent.get("filingDate", [])
    report_dates = recent.get("reportDate", [])
    acc_nums = recent.get("accessionNumber", [])

    rows = []
    for form, fd, rd, an in zip(forms, filing_dates, report_dates, acc_nums):
        if form not in ("10-Q", "10-K"):
            continue
        if fd is None or fd == "":
            continue
        rows.append({
            "ticker": t,
            "form": form,
            "filingDate": fd,
            "reportDate": rd,
            "accessionNumber": an
        })
        if len(rows) >= max_n:
            break

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return pd.DataFrame(columns=["earnings_date","ticker","form","filingDate","reportDate","accessionNumber"])

    df["filingDate"] = pd.to_datetime(df["filingDate"]).dt.tz_localize(None).dt.normalize()
    # 이벤트 날짜 컬럼은 earnings_date로 "형식 통일" (이름만 유지)
    df["earnings_date"] = df["filingDate"]
    df = df.sort_values("earnings_date").drop_duplicates(subset=["ticker","earnings_date"])
    return df

def path_sec_filings(ticker: str) -> Path:
    return RAW_DIR / f"sec_filings_{ticker}.parquet"


# -----------------------------
# 실행
# -----------------------------
sess = requests.Session()

print("1) SEC ticker->CIK 매핑 로딩...")
TICKER_CIK = load_ticker_cik_map(sess)
print("   - mapped tickers:", len(TICKER_CIK))

sec_events_map = {}

for i, tk in enumerate(TICKERS, 1):
    fp = path_sec_filings(tk)
    if fp.exists():
        df_f = pd.read_parquet(fp)
        sec_events_map[tk] = df_f
        print(f"[{i}/{len(TICKERS)}] {tk}: cache load {len(df_f)}")
        continue

    print(f"[{i}/{len(TICKERS)}] {tk}: SEC filings fetch...")
    try:
        df_f = fetch_filings_10q_10k(tk, sess, max_n=400)
    except Exception as e:
        print("   - 실패:", repr(e))
        df_f = pd.DataFrame(columns=["earnings_date","ticker","form","filingDate","reportDate","accessionNumber"])

    df_f.to_parquet(fp, index=False)
    sec_events_map[tk] = df_f
    print(f"   - events: {len(df_f)}")

    # SEC 예의: 과도한 호출 방지(짧게 쉼)
    time.sleep(0.2)

print("\n✅ 셀 4 완료: SEC 기반 이벤트 수집 끝")
print("이제 earnings_map을 SEC 이벤트로 '대체'하겠소.")
earnings_map = sec_events_map


1) SEC ticker->CIK 매핑 로딩...
   - mapped tickers: 10301
[1/5] AAPL: cache load 43
[2/5] MSFT: cache load 25
[3/5] AMZN: cache load 25
[4/5] GOOGL: cache load 14
[5/5] NVDA: cache load 25

✅ 셀 4 완료: SEC 기반 이벤트 수집 끝
이제 earnings_map을 SEC 이벤트로 '대체'하겠소.


In [4]:
# =========================================
# [셀 3-디버그] earnings_map/캐시 상태 점검 + 즉시 재시도
# 목적:
# - events_df=0의 원인이 "실적발표일 0건"인지 즉시 확인
# - 빈 캐시가 굳어졌는지 확인
# =========================================

import os

def show_earnings_status(ticker: str):
    fp = path_earnings(ticker)
    df_e = earnings_map.get(ticker, None)

    print(f"\n=== {ticker} ===")
    print("earnings parquet exists:", fp.exists(), "| path:", fp)
    if fp.exists():
        print("earnings parquet size(bytes):", fp.stat().st_size)

    if df_e is None:
        print("earnings_map entry: None")
        return

    print("earnings_map rows:", len(df_e), "| cols:", list(df_e.columns))
    if len(df_e) > 0:
        display(df_e.head(5))

for tk in TICKERS:
    show_earnings_status(tk)

print("\n[체크] 가격 데이터는 정상인가?")
for tk in TICKERS:
    df_px = prices_map.get(tk)
    if df_px is None:
        print(tk, "prices_map None")
    else:
        print(tk, "price rows:", len(df_px), "range:", df_px.index.min().date(), "~", df_px.index.max().date())



=== AAPL ===
earnings parquet exists: True | path: C:\QP2\data\raw\earnings_AAPL.parquet
earnings parquet size(bytes): 1494
earnings_map rows: 43 | cols: ['ticker', 'form', 'filingDate', 'reportDate', 'accessionNumber', 'earnings_date']


,ticker,form,filingDate,reportDate,accessionNumber,earnings_date
0,AAPL,10-Q,2015-04-28,2015-03-28,0001193125-15-153166,2015-04-28
1,AAPL,10-Q,2015-07-22,2015-06-27,0001193125-15-259935,2015-07-22
2,AAPL,10-K,2015-10-28,2015-09-26,0001193125-15-356351,2015-10-28
3,AAPL,10-Q,2016-01-27,2015-12-26,0001193125-16-439878,2016-01-27
4,AAPL,10-Q,2016-04-27,2016-03-26,0001193125-16-559625,2016-04-27



=== MSFT ===
earnings parquet exists: True | path: C:\QP2\data\raw\earnings_MSFT.parquet
earnings parquet size(bytes): 1494
earnings_map rows: 25 | cols: ['ticker', 'form', 'filingDate', 'reportDate', 'accessionNumber', 'earnings_date']


,ticker,form,filingDate,reportDate,accessionNumber,earnings_date
0,MSFT,10-Q,2019-10-23,2019-09-30,0001564590-19-037549,2019-10-23
1,MSFT,10-Q,2020-01-29,2019-12-31,0001564590-20-002450,2020-01-29
2,MSFT,10-Q,2020-04-29,2020-03-31,0001564590-20-019706,2020-04-29
3,MSFT,10-K,2020-07-30,2020-06-30,0001564590-20-034944,2020-07-30
4,MSFT,10-Q,2020-10-27,2020-09-30,0001564590-20-047996,2020-10-27



=== AMZN ===
earnings parquet exists: True | path: C:\QP2\data\raw\earnings_AMZN.parquet
earnings parquet size(bytes): 1494
earnings_map rows: 25 | cols: ['ticker', 'form', 'filingDate', 'reportDate', 'accessionNumber', 'earnings_date']


...


=== GOOGL ===
earnings parquet exists: True | path: C:\QP2\data\raw\earnings_GOOGL.parquet
earnings parquet size(bytes): 1494
earnings_map rows: 14 | cols: ['ticker', 'form', 'filingDate', 'reportDate', 'accessionNumber', 'earnings_date']


,ticker,form,filingDate,reportDate,accessionNumber,earnings_date
0,GOOGL,10-Q,2022-07-27,2022-06-30,0001652044-22-000071,2022-07-27
1,GOOGL,10-Q,2022-10-26,2022-09-30,0001652044-22-000090,2022-10-26
2,GOOGL,10-K,2023-02-03,2022-12-31,0001652044-23-000016,2023-02-03
3,GOOGL,10-Q,2023-04-26,2023-03-31,0001652044-23-000045,2023-04-26
4,GOOGL,10-Q,2023-07-26,2023-06-30,0001652044-23-000070,2023-07-26



=== NVDA ===
earnings parquet exists: True | path: C:\QP2\data\raw\earnings_NVDA.parquet
earnings parquet size(bytes): 1494
earnings_map rows: 25 | cols: ['ticker', 'form', 'filingDate', 'reportDate', 'accessionNumber', 'earnings_date']


,ticker,form,filingDate,reportDate,accessionNumber,earnings_date
0,NVDA,10-Q,2019-11-14,2019-10-27,0001045810-19-000170,2019-11-14
1,NVDA,10-K,2020-02-20,2020-01-26,0001045810-20-000010,2020-02-20
2,NVDA,10-Q,2020-05-21,2020-04-26,0001045810-20-000065,2020-05-21
3,NVDA,10-Q,2020-08-19,2020-07-26,0001045810-20-000147,2020-08-19
4,NVDA,10-Q,2020-11-18,2020-10-25,0001045810-20-000189,2020-11-18



[체크] 가격 데이터는 정상인가?
AAPL price rows: 2783 range: 2015-01-02 ~ 2026-01-27
MSFT price rows: 2783 range: 2015-01-02 ~ 2026-01-27
AMZN price rows: 2783 range: 2015-01-02 ~ 2026-01-27
GOOGL price rows: 2783 range: 2015-01-02 ~ 2026-01-27
NVDA price rows: 2783 range: 2015-01-02 ~ 2026-01-27


In [5]:
# =========================================
# [셀 4] 이벤트 정렬 + 사전 모멘텀/유지(강화) 지표 계산
# 목적:
# - 실적발표일(E)을 "가격 거래일 캘린더"에 매핑
# - 사전 모멘텀: [E-PRE_LOOKBACK, E-1] 수익률 합/기울기
# - 직전 유지/강화: [E-HOLD_WINDOW, E-1]에서 같은 방향 유지/가속 여부
# 산출:
# - events_df: (ticker, earnings_date, E_trade_date, metrics...)
# =========================================

from dataclasses import dataclass

@dataclass
class EventMetrics:
    mom_pre_sum: float
    mom_pre_slope: float
    mom_hold_sum: float
    mom_hold_slope: float
    keep_direction: int
    accelerate: int
    valid: int
    reason: str


def _nearest_trading_day(target: pd.Timestamp, trading_days: pd.DatetimeIndex) -> pd.Timestamp | None:
    """
    target에 가장 가까운 거래일(인덱스)을 반환하오.
    """
    if target is None or pd.isna(target):
        return None
    target = pd.to_datetime(target).tz_localize(None).normalize()
    if len(trading_days) == 0:
        return None
    # searchsorted로 근접 찾기
    pos = trading_days.searchsorted(target)
    if pos == 0:
        return trading_days[0]
    if pos >= len(trading_days):
        return trading_days[-1]
    before = trading_days[pos - 1]
    after = trading_days[pos]
    # 더 가까운 쪽
    return before if abs((target - before).days) <= abs((after - target).days) else after


def compute_event_metrics_for_one(ticker: str,
                                  df_px: pd.DataFrame,
                                  df_e: pd.DataFrame,
                                  pre_lookback: int = PRE_LOOKBACK,
                                  hold_window: int = HOLD_WINDOW,
                                  min_gap_days: int = MIN_GAP_DAYS,
                                  use_log_ret: bool = USE_LOG_RET) -> pd.DataFrame:
    """
    단일 종목에 대해 이벤트별 지표를 계산하오.
    """
    if df_px is None or len(df_px) == 0:
        return pd.DataFrame()

    px = df_px[PRICE_FIELD].dropna().copy()
    px.index = pd.to_datetime(px.index).tz_localize(None)
    px = px.sort_index()
    trading_days = px.index

    rets = compute_returns(px, use_log=use_log_ret).dropna()
    if len(rets) < (pre_lookback + 5):
        return pd.DataFrame()

    # 실적발표일 목록
    if df_e is None or len(df_e) == 0 or "earnings_date" not in df_e.columns:
        return pd.DataFrame(columns=["ticker", "earnings_date", "E_trade_date"])

    e_dates = pd.to_datetime(df_e["earnings_date"]).dropna().dt.tz_localize(None).dt.normalize().sort_values().unique()
    if len(e_dates) == 0:
        return pd.DataFrame(columns=["ticker", "earnings_date", "E_trade_date"])

    # 이벤트 간격 필터(너무 촘촘하면 중첩 제거)
    e_dates_f = []
    last = None
    for d in e_dates:
        if last is None:
            e_dates_f.append(d); last = d
        else:
            if (d - last).days >= min_gap_days:
                e_dates_f.append(d); last = d

    rows = []
    for ed in e_dates_f:
        E = _nearest_trading_day(ed, trading_days)
        if E is None:
            continue

        # E가 수익률 인덱스에 있어야 계산 가능(일반적으로 다음날 diff로 인해 첫날 결측)
        if E not in trading_days:
            continue

        # E-1 거래일 인덱스 위치
        idx_E = trading_days.get_loc(E)
        if idx_E < 2:
            continue

        # 모멘텀 계산 구간(거래일 기준)
        # pre: [E-pre_lookback, E-1]
        start_pre_idx = idx_E - pre_lookback
        end_pre_idx   = idx_E - 1
        # hold: [E-hold_window, E-1]
        start_hold_idx = idx_E - hold_window
        end_hold_idx   = idx_E - 1

        if start_pre_idx < 1 or start_hold_idx < 1:
            # 충분한 과거가 없음
            rows.append({
                "ticker": ticker,
                "earnings_date": ed,
                "E_trade_date": E,
                "valid": 0,
                "reason": "not_enough_history"
            })
            continue

        win_pre_days  = trading_days[start_pre_idx: end_pre_idx + 1]
        win_hold_days = trading_days[start_hold_idx: end_hold_idx + 1]

        # 수익률은 날짜 기준으로 slicing
        r_pre  = rets.loc[win_pre_days].dropna()
        r_hold = rets.loc[win_hold_days].dropna()

        if len(r_pre) < pre_lookback * 0.8 or len(r_hold) < hold_window * 0.8:
            rows.append({
                "ticker": ticker,
                "earnings_date": ed,
                "E_trade_date": E,
                "valid": 0,
                "reason": "too_many_missing"
            })
            continue

        # 모멘텀(합): 단순히 누적수익률(로그면 합이 누적 로그수익률)
        mom_pre_sum = float(r_pre.sum())
        mom_hold_sum = float(r_hold.sum())

        # 기울기(slope): 시간축 0..n-1에 대해 OLS slope
        def slope_of(series: pd.Series) -> float:
            y = series.values.astype(float)
            x = np.arange(len(y), dtype=float)
            x = x - x.mean()
            denom = (x**2).sum()
            if denom == 0:
                return 0.0
            return float((x * (y - y.mean())).sum() / denom)

        # 누적수익률 곡선의 기울기(더 직관적)
        cum_pre = r_pre.cumsum()
        cum_hold = r_hold.cumsum()
        mom_pre_slope = slope_of(cum_pre)
        mom_hold_slope = slope_of(cum_hold)

        # 방향 유지(keep_direction):
        # - 사전 모멘텀이 +면 hold 구간도 +인지
        # - 사전 모멘텀이 -면 hold 구간도 -인지
        #   (엄밀히는 slope로 방향을 잡되, sum도 함께 고려)
        pre_sign = np.sign(mom_pre_slope) if mom_pre_slope != 0 else np.sign(mom_pre_sum)
        hold_sign = np.sign(mom_hold_slope) if mom_hold_slope != 0 else np.sign(mom_hold_sum)
        keep_direction = int((pre_sign != 0) and (pre_sign == hold_sign))

        # 강화(accelerate):
        # - hold의 평균 일간 모멘텀(또는 slope)이 pre 대비 커졌는지
        pre_avg = mom_pre_sum / len(r_pre)
        hold_avg = mom_hold_sum / len(r_hold)
        accelerate = int(keep_direction == 1 and abs(hold_avg) > abs(pre_avg))

        rows.append({
            "ticker": ticker,
            "earnings_date": ed,
            "E_trade_date": E,
            "mom_pre_sum": mom_pre_sum,
            "mom_pre_slope": mom_pre_slope,
            "mom_hold_sum": mom_hold_sum,
            "mom_hold_slope": mom_hold_slope,
            "keep_direction": keep_direction,
            "accelerate": accelerate,
            "pre_avg": pre_avg,
            "hold_avg": hold_avg,
            "valid": 1,
            "reason": "ok"
        })

    df_out = pd.DataFrame(rows)
    if len(df_out) == 0:
        return df_out

    # 정리: 날짜형/정렬
    df_out["earnings_date"] = pd.to_datetime(df_out["earnings_date"])
    df_out["E_trade_date"] = pd.to_datetime(df_out["E_trade_date"])
    df_out = df_out.sort_values(["ticker", "E_trade_date"]).reset_index(drop=True)
    return df_out


# ---------------------------------
# 실행: 모든 티커 이벤트 지표 계산
# ---------------------------------
events_list = []
for tk in TICKERS:
    df_px = prices_map.get(tk)
    df_e  = earnings_map.get(tk)
    df_ev = compute_event_metrics_for_one(
        ticker=tk,
        df_px=df_px,
        df_e=df_e,
        pre_lookback=PRE_LOOKBACK,
        hold_window=HOLD_WINDOW,
        min_gap_days=MIN_GAP_DAYS,
        use_log_ret=USE_LOG_RET
    )
    if df_ev is not None and len(df_ev) > 0:
        events_list.append(df_ev)

events_df = pd.concat(events_list, ignore_index=True) if len(events_list) > 0 else pd.DataFrame()

print("✅ events_df 생성 완료")
print(" - rows:", len(events_df))
if len(events_df) > 0:
    print(" - valid rows:", int((events_df["valid"] == 1).sum()) if "valid" in events_df.columns else "N/A")
    display(events_df.head(10))

# ---------------------------------
# 빠른 요약 통계(가설의 1차 검정)
# ---------------------------------
if len(events_df) > 0 and "valid" in events_df.columns:
    dfv = events_df[events_df["valid"] == 1].copy()
    if len(dfv) > 0:
        keep_rate = dfv["keep_direction"].mean()
        acc_rate = dfv["accelerate"].mean()
        print("\n[요약]")
        print(f" - 방향 유지 비율(keep_direction): {keep_rate:.3f}")
        print(f" - 강화 비율(accelerate):          {acc_rate:.3f}")
        print(" - 표본 수(valid):", len(dfv))
    else:
        print("\n[요약] valid 표본이 없소. (실적발표일/가격 매핑을 점검하시오.)")
else:
    print("\n[요약] events_df가 비었소. (실적발표일 수집이 빈약했을 가능성이 큼)")


✅ events_df 생성 완료
 - rows: 132
 - valid rows: 132


,ticker,earnings_date,E_trade_date,mom_pre_sum,mom_pre_slope,mom_hold_sum,mom_hold_slope,keep_direction,accelerate,pre_avg,hold_avg,valid,reason
0,AAPL,2015-04-28,2015-04-28,0.073499,0.002016,0.038814,0.010130,1,1,0.003675,0.007763,1,ok
1,AAPL,2015-07-22,2015-07-22,0.024308,0.001021,0.040105,0.008836,1,1,0.001215,0.008021,1,ok
2,AAPL,2015-10-28,2015-10-28,0.049113,0.002876,0.006832,0.001193,1,0,0.002456,0.001366,1,ok
3,AAPL,2016-01-27,2016-01-27,-0.077339,-0.004757,0.033870,0.009714,0,0,-0.003867,0.006774,1,ok
4,AAPL,2016-04-27,2016-04-27,-0.031413,-0.002471,-0.024237,-0.006102,1,1,-0.001571,-0.004847,1,ok
5,AAPL,2016-07-27,2016-07-27,0.049079,0.002614,-0.032566,-0.008818,0,0,0.002454,-0.006513,1,ok
6,AAPL,2016-10-26,2016-10-26,0.044617,0.002736,0.006618,0.002423,1,0,0.002231,0.001324,1,ok
7,AAPL,2017-02-01,2017-02-01,0.046642,0.002362,0.011437,-0.001126,0,0,0.002332,0.002287,1,ok
8,AAPL,2017-05-03,2017-05-03,0.026169,0.000797,0.020409,0.007183,1,1,0.001308,0.004082,1,ok
9,AAPL,2017-08-02,2017-08-02,0.044634,0.002833,-0.017769,-0.005717,0,0,0.002232,-0.003554,1,ok



[요약]
 - 방향 유지 비율(keep_direction): 0.614
 - 강화 비율(accelerate):          0.417
 - 표본 수(valid): 132


In [6]:
# =========================================
# [셀 5] Run-All용 통합 엔진: 이벤트 소스 자동선택 → events_df 생성 → 저장/요약
# 목적:
# - yfinance earnings가 0건이면 자동으로 SEC 10-Q/10-K filingDate로 대체
# - 최종 events_df를 만들어 parquet/csv로 저장
# 전제:
# - [셀1] 경로/파라미터/compute_returns/path_* 정의되어 있음
# - [셀2] prices_map이 채워져 있음 (가격은 정상이라 했으니 OK)
# - [셀3]의 compute_event_metrics_for_one / _nearest_trading_day 등이 있으면 재사용,
#   없으면 이 셀에서 최소 기능을 다시 정의함(하단 "SAFE DEFINE" 참고)
# =========================================

import time
import json
import requests

# -----------------------------
# 0) SAFE DEFINE: (이전 셀 함수가 없을 때 대비)
# -----------------------------
def _ensure_defined():
    g = globals()

    if "compute_event_metrics_for_one" in g and callable(g["compute_event_metrics_for_one"]):
        return  # 이미 정의되어 있으면 그대로 사용

    # --- 최소 재정의(셀3 로직 축약판) ---
    def _nearest_trading_day(target: pd.Timestamp, trading_days: pd.DatetimeIndex):
        if target is None or pd.isna(target) or len(trading_days) == 0:
            return None
        target = pd.to_datetime(target).tz_localize(None).normalize()
        pos = trading_days.searchsorted(target)
        if pos == 0:
            return trading_days[0]
        if pos >= len(trading_days):
            return trading_days[-1]
        before = trading_days[pos - 1]
        after = trading_days[pos]
        return before if abs((target - before).days) <= abs((after - target).days) else after

    def compute_event_metrics_for_one(ticker, df_px, df_e,
                                      pre_lookback=PRE_LOOKBACK,
                                      hold_window=HOLD_WINDOW,
                                      min_gap_days=MIN_GAP_DAYS,
                                      use_log_ret=USE_LOG_RET):
        if df_px is None or len(df_px) == 0:
            return pd.DataFrame()

        px = df_px[PRICE_FIELD].dropna().copy()
        px.index = pd.to_datetime(px.index).tz_localize(None)
        px = px.sort_index()
        trading_days = px.index

        rets = compute_returns(px, use_log=use_log_ret).dropna()
        if len(rets) < (pre_lookback + 5):
            return pd.DataFrame()

        if df_e is None or len(df_e) == 0 or "earnings_date" not in df_e.columns:
            return pd.DataFrame(columns=["ticker", "earnings_date", "E_trade_date"])

        e_dates = pd.to_datetime(df_e["earnings_date"]).dropna().dt.tz_localize(None).dt.normalize().sort_values().unique()
        if len(e_dates) == 0:
            return pd.DataFrame(columns=["ticker", "earnings_date", "E_trade_date"])

        # 간격 필터
        e_dates_f = []
        last = None
        for d in e_dates:
            if last is None or (d - last).days >= min_gap_days:
                e_dates_f.append(d)
                last = d

        def slope_of(series: pd.Series) -> float:
            y = series.values.astype(float)
            x = np.arange(len(y), dtype=float)
            x = x - x.mean()
            denom = (x**2).sum()
            if denom == 0:
                return 0.0
            return float((x * (y - y.mean())).sum() / denom)

        rows = []
        for ed in e_dates_f:
            E = _nearest_trading_day(ed, trading_days)
            if E is None:
                continue
            idx_E = trading_days.get_loc(E)
            start_pre = idx_E - pre_lookback
            start_hold = idx_E - hold_window
            end_ = idx_E - 1
            if start_pre < 1 or start_hold < 1:
                rows.append({"ticker": ticker, "earnings_date": ed, "E_trade_date": E, "valid": 0, "reason": "not_enough_history"})
                continue

            win_pre = trading_days[start_pre:end_+1]
            win_hold = trading_days[start_hold:end_+1]
            r_pre = rets.loc[win_pre].dropna()
            r_hold = rets.loc[win_hold].dropna()

            if len(r_pre) < pre_lookback * 0.8 or len(r_hold) < hold_window * 0.8:
                rows.append({"ticker": ticker, "earnings_date": ed, "E_trade_date": E, "valid": 0, "reason": "too_many_missing"})
                continue

            mom_pre_sum = float(r_pre.sum())
            mom_hold_sum = float(r_hold.sum())
            cum_pre = r_pre.cumsum()
            cum_hold = r_hold.cumsum()
            mom_pre_slope = slope_of(cum_pre)
            mom_hold_slope = slope_of(cum_hold)

            pre_sign = np.sign(mom_pre_slope) if mom_pre_slope != 0 else np.sign(mom_pre_sum)
            hold_sign = np.sign(mom_hold_slope) if mom_hold_slope != 0 else np.sign(mom_hold_sum)
            keep_direction = int((pre_sign != 0) and (pre_sign == hold_sign))

            pre_avg = mom_pre_sum / len(r_pre)
            hold_avg = mom_hold_sum / len(r_hold)
            accelerate = int(keep_direction == 1 and abs(hold_avg) > abs(pre_avg))

            rows.append({
                "ticker": ticker,
                "earnings_date": ed,
                "E_trade_date": E,
                "mom_pre_sum": mom_pre_sum,
                "mom_pre_slope": mom_pre_slope,
                "mom_hold_sum": mom_hold_sum,
                "mom_hold_slope": mom_hold_slope,
                "pre_avg": pre_avg,
                "hold_avg": hold_avg,
                "keep_direction": keep_direction,
                "accelerate": accelerate,
                "valid": 1,
                "reason": "ok"
            })

        out = pd.DataFrame(rows)
        if len(out) == 0:
            return out
        out["earnings_date"] = pd.to_datetime(out["earnings_date"])
        out["E_trade_date"] = pd.to_datetime(out["E_trade_date"])
        return out.sort_values(["ticker","E_trade_date"]).reset_index(drop=True)

    g["_nearest_trading_day"] = _nearest_trading_day
    g["compute_event_metrics_for_one"] = compute_event_metrics_for_one


_ensure_defined()


# -----------------------------
# 1) 이벤트 소스 자동선택: yfinance earnings → 없으면 SEC filings
# -----------------------------
SEC_TICKER_CIK_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_SUBMISSIONS_URL = "https://data.sec.gov/submissions/CIK{cik10}.json"

# SEC는 User-Agent가 거의 필수요. 네 것으로 바꾸시오(이메일 포함 권장).
SEC_USER_AGENT = "QP Research (add-your-email@example.com)"

def sec_get(url: str, session: requests.Session) -> dict:
    r = session.get(url, headers={"User-Agent": SEC_USER_AGENT, "Accept-Encoding": "gzip, deflate"})
    r.raise_for_status()
    return r.json()

def load_ticker_cik_map(session: requests.Session) -> dict:
    data = sec_get(SEC_TICKER_CIK_URL, session)
    mp = {}
    for _, v in data.items():
        t = str(v.get("ticker", "")).upper()
        cik = int(v.get("cik_str"))
        if t:
            mp[t] = cik
    return mp

def fetch_filings_10q_10k(ticker: str, session: requests.Session, ticker_cik: dict, max_n: int = 400) -> pd.DataFrame:
    t = ticker.upper()
    cik = ticker_cik.get(t)
    if cik is None:
        return pd.DataFrame(columns=["earnings_date","ticker","form","filingDate","reportDate","accessionNumber"])

    cik10 = str(cik).zfill(10)
    js = sec_get(SEC_SUBMISSIONS_URL.format(cik10=cik10), session)
    recent = js.get("filings", {}).get("recent", {})

    forms = recent.get("form", [])
    filing_dates = recent.get("filingDate", [])
    report_dates = recent.get("reportDate", [])
    acc_nums = recent.get("accessionNumber", [])

    rows = []
    for form, fd, rd, an in zip(forms, filing_dates, report_dates, acc_nums):
        if form not in ("10-Q","10-K"):
            continue
        if not fd:
            continue
        rows.append({
            "ticker": t,
            "form": form,
            "filingDate": fd,
            "reportDate": rd,
            "accessionNumber": an
        })
        if len(rows) >= max_n:
            break

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return pd.DataFrame(columns=["earnings_date","ticker","form","filingDate","reportDate","accessionNumber"])

    df["filingDate"] = pd.to_datetime(df["filingDate"]).dt.tz_localize(None).dt.normalize()
    df["earnings_date"] = df["filingDate"]  # 이름 통일
    df = df.sort_values("earnings_date").drop_duplicates(subset=["ticker","earnings_date"])
    return df

def path_sec_filings(ticker: str) -> Path:
    return RAW_DIR / f"sec_filings_{ticker}.parquet"


def build_events_map_auto(tickers: list[str]) -> tuple[dict, str]:
    """
    1) yfinance earnings 캐시/맵에서 유효 이벤트가 있으면 사용
    2) 없으면 SEC filings로 생성
    반환: (events_map, source_name)
    """
    # (A) yfinance 쪽 확인: rows>0 종목이 1개라도 있으면 그걸로 간주
    yf_ok = False
    yf_map = {}

    for tk in tickers:
        df_e = None
        # 우선 메모리에 있으면
        if "earnings_map" in globals() and isinstance(earnings_map, dict):
            df_e = earnings_map.get(tk)

        # 없거나 빈 경우 parquet에서 직접 읽어봄
        if df_e is None or len(df_e) == 0:
            fp = path_earnings(tk)
            if fp.exists():
                try:
                    df_e = pd.read_parquet(fp)
                except Exception:
                    df_e = None

        if df_e is not None and len(df_e) > 0:
            yf_ok = True

        yf_map[tk] = df_e if df_e is not None else pd.DataFrame(columns=["earnings_date","ticker"])

    if yf_ok:
        return yf_map, "yfinance_earnings"

    # (B) SEC로 대체
    sess = requests.Session()
    ticker_cik = load_ticker_cik_map(sess)

    sec_map = {}
    for i, tk in enumerate(tickers, 1):
        fp = path_sec_filings(tk)
        if fp.exists():
            df_f = pd.read_parquet(fp)
            sec_map[tk] = df_f
            continue

        df_f = fetch_filings_10q_10k(tk, sess, ticker_cik, max_n=400)
        df_f.to_parquet(fp, index=False)
        sec_map[tk] = df_f

        time.sleep(0.2)

    return sec_map, "sec_filings_10q10k"


events_map, EVENTS_SOURCE = build_events_map_auto(TICKERS)
print("✅ 이벤트 소스 선택:", EVENTS_SOURCE)


# -----------------------------
# 2) events_df 생성(모멘텀/유지/강화)
# -----------------------------
events_list = []
for tk in TICKERS:
    df_px = prices_map.get(tk)
    df_e  = events_map.get(tk)
    if df_e is None or len(df_e) == 0:
        continue
    df_ev = compute_event_metrics_for_one(
        ticker=tk,
        df_px=df_px,
        df_e=df_e,
        pre_lookback=PRE_LOOKBACK,
        hold_window=HOLD_WINDOW,
        min_gap_days=MIN_GAP_DAYS,
        use_log_ret=USE_LOG_RET
    )
    if df_ev is not None and len(df_ev) > 0:
        df_ev["events_source"] = EVENTS_SOURCE
        events_list.append(df_ev)

events_df = pd.concat(events_list, ignore_index=True) if len(events_list) > 0 else pd.DataFrame()
print("✅ events_df rows:", len(events_df))

# -----------------------------
# 3) 저장 + 요약
# -----------------------------
out_parquet = PROC_DIR / f"events_df_{EVENTS_SOURCE}_pre{PRE_LOOKBACK}_hold{HOLD_WINDOW}.parquet"
out_csv     = OUT_DIR  / f"events_df_{EVENTS_SOURCE}_pre{PRE_LOOKBACK}_hold{HOLD_WINDOW}.csv"

if len(events_df) > 0:
    events_df.to_parquet(out_parquet, index=False)
    events_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    dfv = events_df[events_df.get("valid", 1) == 1].copy() if "valid" in events_df.columns else events_df.copy()
    if len(dfv) > 0 and "keep_direction" in dfv.columns:
        keep_rate = float(dfv["keep_direction"].mean())
        acc_rate  = float(dfv["accelerate"].mean()) if "accelerate" in dfv.columns else np.nan
        print("\n[요약]")
        print(" - 표본 수(valid):", len(dfv))
        print(f" - 방향 유지 비율(keep_direction): {keep_rate:.3f}")
        print(f" - 강화 비율(accelerate):          {acc_rate:.3f}")
        print(" - 저장:", out_parquet)
        print(" - 저장:", out_csv)
    else:
        print("\n[요약] events_df는 있으나 지표 컬럼이 비정상/부족하오. 컬럼을 확인하시오.")
        print("columns:", list(events_df.columns))
else:
    print("\n[경고] events_df가 비었소.")
    print(" - SEC_USER_AGENT 미설정/차단 가능, 또는 종목/네트워크 문제 가능")
    print(" - 우선 각 ticker별 events_map[ticker] rows를 확인하시오.")


✅ 이벤트 소스 선택: yfinance_earnings
✅ events_df rows: 132

[요약]
 - 표본 수(valid): 132
 - 방향 유지 비율(keep_direction): 0.614
 - 강화 비율(accelerate):          0.417
 - 저장: C:\QP2\data\processed\events_df_yfinance_earnings_pre20_hold5.parquet
 - 저장: C:\QP2\outputs\events_df_yfinance_earnings_pre20_hold5.csv


In [7]:
# =========================================
# [셀 6] 규칙 검정: 엄격한 유지/강화 정의 + 통계검정(부트스트랩)
# 목적:
# - "발표(이벤트) 직전까지 모멘텀 유지/강화"가 우연인지 구조인지 검정
# 입력:
# - prices_map, events_map, events_df (셀5 산출)
# 산출:
# - test_df: 이벤트별 엄격 지표
# - summary: 전체/종목별 유지율 및 유의성(부트스트랩)
# =========================================

import numpy as np
import pandas as pd

# -----------------------------
# 0) 유틸: OLS slope & t-stat
# -----------------------------
def ols_slope_tstat(y: np.ndarray) -> tuple[float, float]:
    """
    y ~ a + b*x (x=0..n-1) 에서 b와 t-stat 반환하오.
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n < 3:
        return 0.0, 0.0

    x = np.arange(n, dtype=float)
    X = np.column_stack([np.ones(n), x])
    # beta = (X'X)^-1 X'y
    XtX = X.T @ X
    try:
        beta = np.linalg.solve(XtX, X.T @ y)
    except np.linalg.LinAlgError:
        return 0.0, 0.0

    yhat = X @ beta
    resid = y - yhat
    dof = n - 2
    s2 = (resid @ resid) / dof if dof > 0 else 0.0

    # var(beta) = s2 * (X'X)^-1
    try:
        XtX_inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        return float(beta[1]), 0.0

    se_b = np.sqrt(s2 * XtX_inv[1, 1]) if s2 >= 0 else 0.0
    tstat = float(beta[1] / se_b) if se_b > 0 else 0.0
    return float(beta[1]), tstat


# -----------------------------
# 1) 엄격 정의 3종
# -----------------------------
def strict_keep_definitions(px: pd.Series, E_trade: pd.Timestamp,
                            pre_lookback: int = PRE_LOOKBACK,
                            hold_window: int = HOLD_WINDOW) -> dict | None:
    """
    한 이벤트에 대해 "유지/강화"를 3가지 엄격 정의로 계산하오.
    반환 dict에는 flag_keep_*, flag_acc_* 등이 포함됨.
    """
    px = px.dropna().astype(float).copy()
    px.index = pd.to_datetime(px.index).tz_localize(None)
    px = px.sort_index()

    if E_trade not in px.index:
        return None

    idx_E = px.index.get_loc(E_trade)
    if idx_E - pre_lookback < 1 or idx_E - hold_window < 1:
        return None

    # 구간 종가
    pre_days = px.index[idx_E - pre_lookback: idx_E]         # E 포함 X, E-... ~ E-1
    hold_days = px.index[idx_E - hold_window: idx_E]         # E 포함 X, E-hold ~ E-1

    pre_px = px.loc[pre_days]
    hold_px = px.loc[hold_days]

    # 수익률
    pre_r = compute_returns(pre_px, USE_LOG_RET).dropna()
    hold_r = compute_returns(hold_px, USE_LOG_RET).dropna()
    if len(pre_r) < pre_lookback * 0.8 or len(hold_r) < hold_window * 0.8:
        return None

    # 사전 추세 방향: 누적수익률 slope sign
    pre_cum = pre_r.cumsum().values
    hold_cum = hold_r.cumsum().values
    pre_slope, pre_t = ols_slope_tstat(pre_cum)
    hold_slope, hold_t = ols_slope_tstat(hold_cum)

    pre_sign = np.sign(pre_slope) if pre_slope != 0 else np.sign(pre_cum[-1])
    hold_sign = np.sign(hold_slope) if hold_slope != 0 else np.sign(hold_cum[-1])

    # 정의 A: 방향 일치 + hold 구간 음의 반전일 비중 제한
    # - pre_sign과 반대 수익률 일수 비중이 40% 이하면 "유지"
    opp = (np.sign(hold_r.values) == -pre_sign).mean() if pre_sign != 0 else 1.0
    keep_A = int((pre_sign != 0) and (hold_sign == pre_sign) and (opp <= 0.4))

    # 정의 B: drawdown 제한(hold 구간 내 누적곡선 최대낙폭이 임계치 이하)
    # - 같은 방향으로 '흔들리되 무너지지' 않았는가
    cum = hold_r.cumsum()
    if pre_sign > 0:
        peak = cum.cummax()
        dd = (cum - peak).min()  # 음수
    elif pre_sign < 0:
        trough = cum.cummin()
        dd = (trough - cum).min()  # 음수(상승 반전 폭을 낙폭으로 간주)
    else:
        dd = -np.inf
    # 임계치: hold_window 동안 -2% 로그수익률 정도(대략). 필요시 조정.
    DD_TH = -0.02
    keep_B = int((pre_sign != 0) and (hold_sign == pre_sign) and (dd >= DD_TH))

    # 정의 C: 기울기 유의성(hold slope t-stat이 같은 방향으로 충분히 큼)
    # - "유지"는 t-stat 절대값이 크고 방향이 같아야 함
    T_TH = 1.5
    keep_C = int((pre_sign != 0) and (hold_sign == pre_sign) and (hold_t * pre_sign >= T_TH))

    # 강화: pre 대비 hold의 평균 일수익률(절대값)이 더 크면 강화로 간주
    pre_avg = float(pre_r.mean())
    hold_avg = float(hold_r.mean())
    acc = int((pre_sign != 0) and (np.sign(hold_avg) == pre_sign) and (abs(hold_avg) > abs(pre_avg)))

    return {
        "pre_slope": pre_slope, "pre_t": pre_t,
        "hold_slope": hold_slope, "hold_t": hold_t,
        "pre_avg": pre_avg, "hold_avg": hold_avg,
        "opp_ratio_hold": float(opp),
        "hold_dd": float(dd),
        "keep_A": keep_A,
        "keep_B": keep_B,
        "keep_C": keep_C,
        "acc_strict": acc,
    }


# -----------------------------
# 2) 이벤트별 엄격 지표 계산
# -----------------------------
if len(events_df) == 0:
    raise RuntimeError("events_df가 비었소. 셀5가 제대로 돌았는지부터 확인하시오.")

rows = []
for _, r in events_df.iterrows():
    tk = r["ticker"]
    E_trade = pd.to_datetime(r["E_trade_date"])
    df_px = prices_map.get(tk)
    if df_px is None or len(df_px) == 0:
        continue
    px = df_px[PRICE_FIELD].copy()

    m = strict_keep_definitions(px, E_trade, PRE_LOOKBACK, HOLD_WINDOW)
    if m is None:
        continue

    out = {**{k: r.get(k) for k in ["ticker", "earnings_date", "E_trade_date", "events_source"] if k in r.index},
           **m}
    rows.append(out)

test_df = pd.DataFrame(rows)
print("✅ test_df rows:", len(test_df))
if len(test_df) > 0:
    display(test_df.head(10))


# -----------------------------
# 3) 요약: 유지율/강화율
# -----------------------------
def summarize(df: pd.DataFrame, label: str):
    if len(df) == 0:
        print(f"\n[{label}] 표본 없음")
        return
    print(f"\n[{label}] n={len(df)}")
    for k in ["keep_A", "keep_B", "keep_C", "acc_strict"]:
        if k in df.columns:
            print(f" - {k}: {df[k].mean():.3f}")

summarize(test_df, "전체")


# -----------------------------
# 4) 부트스트랩 유의성(우연 대비)
# 아이디어:
# - 고정된 이벤트 날짜 대신 같은 수만큼 "무작위 거래일"을 뽑아
#   keep_* 비율이 얼마나 자주 현재 이상이 나오는지(p-value) 계산
# -----------------------------
BOOT_N = 300  # 빠르게 300회; 필요시 1000으로 늘리시오.
rng = np.random.default_rng(42)

def bootstrap_null_keep_rate(ticker: str, n_events: int) -> dict:
    """
    ticker별로 이벤트 수 n_events에 맞춰 무작위 날짜를 뽑아 keep율 분포를 만들겠소.
    """
    df_px = prices_map.get(ticker)
    if df_px is None or len(df_px) == 0:
        return {}
    px = df_px[PRICE_FIELD].dropna().copy()
    td = pd.to_datetime(px.index).tz_localize(None)
    td = td.sort_values()

    # 이벤트 윈도우를 고려해 샘플 가능한 날짜 범위 제한
    min_idx = PRE_LOOKBACK + 2
    max_idx = len(td) - 2
    if max_idx <= min_idx + 10:
        return {}

    # 실제 계산 가능한 날짜 후보
    candidates = td[min_idx:max_idx]

    keepA_rates, keepB_rates, keepC_rates = [], [], []
    for _ in range(BOOT_N):
        sampled = rng.choice(candidates, size=n_events, replace=False) if len(candidates) >= n_events else rng.choice(candidates, size=n_events, replace=True)
        # 각 샘플 날짜에 대해 keep 계산
        ka = kb = kc = 0
        cnt = 0
        for E in sampled:
            m = strict_keep_definitions(px, pd.to_datetime(E), PRE_LOOKBACK, HOLD_WINDOW)
            if m is None:
                continue
            cnt += 1
            ka += m["keep_A"]
            kb += m["keep_B"]
            kc += m["keep_C"]
        if cnt == 0:
            continue
        keepA_rates.append(ka / cnt)
        keepB_rates.append(kb / cnt)
        keepC_rates.append(kc / cnt)

    def arr_stats(a):
        a = np.array(a, dtype=float)
        return {"mean": float(np.mean(a)), "p95": float(np.quantile(a, 0.95)), "dist": a}

    out = {}
    if len(keepA_rates) > 10:
        out["A"] = arr_stats(keepA_rates)
    if len(keepB_rates) > 10:
        out["B"] = arr_stats(keepB_rates)
    if len(keepC_rates) > 10:
        out["C"] = arr_stats(keepC_rates)
    return out


# 실제 유지율 대비 p-value(귀무: 무작위에서도 이 정도 나옴?)
p_rows = []
for tk, gdf in test_df.groupby("ticker"):
    n_ev = len(gdf)
    obsA = gdf["keep_A"].mean()
    obsB = gdf["keep_B"].mean()
    obsC = gdf["keep_C"].mean()

    null = bootstrap_null_keep_rate(tk, n_ev)
    if not null:
        continue

    def pval(obs, dist):
        dist = np.asarray(dist, dtype=float)
        return float((dist >= obs).mean())

    row = {"ticker": tk, "n_events": n_ev,
           "obs_keep_A": obsA, "obs_keep_B": obsB, "obs_keep_C": obsC}

    if "A" in null:
        row["null_mean_A"] = null["A"]["mean"]
        row["null_p95_A"] = null["A"]["p95"]
        row["pval_A"] = pval(obsA, null["A"]["dist"])
    if "B" in null:
        row["null_mean_B"] = null["B"]["mean"]
        row["null_p95_B"] = null["B"]["p95"]
        row["pval_B"] = pval(obsB, null["B"]["dist"])
    if "C" in null:
        row["null_mean_C"] = null["C"]["mean"]
        row["null_p95_C"] = null["C"]["p95"]
        row["pval_C"] = pval(obsC, null["C"]["dist"])

    p_rows.append(row)

pvals_df = pd.DataFrame(p_rows).sort_values("ticker").reset_index(drop=True)
print("\n✅ 부트스트랩 p-value 요약(pvals_df)")
display(pvals_df)


✅ test_df rows: 132


,ticker,earnings_date,E_trade_date,events_source,pre_slope,pre_t,hold_slope,hold_t,pre_avg,hold_avg,opp_ratio_hold,hold_dd,keep_A,keep_B,keep_C,acc_strict
0,AAPL,2015-04-28,2015-04-28,yfinance_earnings,0.002269,5.122862,0.009725,4.952430,0.002553,0.011059,0.00,0.000000,1,1,1,1
1,AAPL,2015-07-22,2015-07-22,yfinance_earnings,0.001275,1.408474,0.007057,1.750251,0.001519,0.007629,0.25,-0.010045,1,1,1,1
2,AAPL,2015-10-28,2015-10-28,yfinance_earnings,0.003053,5.581184,-0.005721,-0.666698,0.001990,0.001730,0.50,-0.038784,0,0,0,0
3,AAPL,2016-01-27,2016-01-27,yfinance_earnings,-0.004497,-3.693350,0.009309,0.929216,-0.003478,0.008132,0.50,-0.051802,0,0,0,0
4,AAPL,2016-04-27,2016-04-27,yfinance_earnings,-0.002738,-4.271064,-0.005191,-7.640280,-0.002564,-0.006573,0.00,0.000000,1,1,1,1
5,AAPL,2016-07-27,2016-07-27,yfinance_earnings,0.002396,4.644683,-0.009792,-11.157907,0.001704,-0.008367,1.00,-0.028151,0,0,0,0
6,AAPL,2016-10-26,2016-10-26,yfinance_earnings,0.002945,8.817719,0.003931,2.118146,0.001950,0.002400,0.50,-0.003938,0,1,1,1
7,AAPL,2017-02-01,2017-02-01,yfinance_earnings,0.002248,9.027153,-0.001718,-3.958196,0.002305,-0.001089,0.50,-0.004932,0,0,0,0
8,AAPL,2017-05-03,2017-05-03,yfinance_earnings,0.001114,2.443713,0.009682,3.542879,0.000987,0.006577,0.25,-0.000974,1,1,1,1
9,AAPL,2017-08-02,2017-08-02,yfinance_earnings,0.002765,5.090075,-0.001534,-0.577751,0.002133,-0.005618,0.75,-0.012229,0,0,0,0



[전체] n=132
 - keep_A: 0.326
 - keep_B: 0.477
 - keep_C: 0.341
 - acc_strict: 0.409

✅ 부트스트랩 p-value 요약(pvals_df)


,ticker,n_events,obs_keep_A,obs_keep_B,obs_keep_C,null_mean_A,null_p95_A,pval_A,null_mean_B,null_p95_B,pval_B,null_mean_C,null_p95_C,pval_C
0,AAPL,43,0.279070,0.418605,0.302326,0.352791,0.488372,0.873333,0.537364,0.674419,0.956667,0.357287,0.488372,0.786667
1,AMZN,25,0.160000,0.440000,0.320000,0.336533,0.520000,0.986667,0.515733,0.680000,0.830000,0.324667,0.480000,0.593333
2,GOOGL,14,0.785714,0.857143,0.571429,0.336905,0.571429,0.003333,0.506429,0.714286,0.003333,0.334524,0.571429,0.056667
3,MSFT,25,0.280000,0.400000,0.280000,0.321333,0.480000,0.726667,0.531600,0.680000,0.940000,0.330533,0.480000,0.746667
4,NVDA,25,0.360000,0.480000,0.360000,0.333200,0.480000,0.463333,0.441867,0.600000,0.433333,0.331067,0.480000,0.470000
